In [1]:
%%writefile customers.csv
customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

Writing customers.csv


In [2]:
%%writefile usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210

Writing usage.csv


In [3]:
%%writefile payments.csv
payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

Writing payments.csv


In [4]:
%%writefile plans.json
[
{
"plan_id": "P101",
"plan_name": "Smart Basic",
"monthly_fee": 499,
"data_limit_gb": 50,
"features": {
"unlimited_calls": true,
"ott_included": false,
"roaming": "National"
}
},
{
"plan_id": "P102",
"plan_name": "Smart Plus",
"monthly_fee": 799,
"data_limit_gb": 75,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "National"
}
},
{
"plan_id": "P103",
"plan_name": "Budget Saver",
"monthly_fee": 299,
"data_limit_gb": 25,
"features": {
"unlimited_calls": false,
"ott_included": false,
"roaming": null
}
},
{
"plan_id": "P104",
"plan_name": "Premium Max",
"monthly_fee": 1199,
"data_limit_gb": 100,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "International"
}
}
]

Writing plans.json


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Telecom Analytics ETL").getOrCreate()

In [6]:
customers_df = spark.read.csv("customers.csv", header=True,inferSchema=True)
usage_df = spark.read.csv(
    "usage.csv",
    header=True,
    inferSchema=True
)

payments_df = spark.read.csv(
    "payments.csv",
    header=True,
    inferSchema=True
)

plans_df = spark.read.json(
    "plans.json",
    multiLine=True
)

In [7]:
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [8]:
print(customers_df.count())
print(usage_df.count())
print(payments_df.count())
print(plans_df.count())

12
15
15
4


In [9]:
customers_df.write.mode("overwrite").parquet("bronze/customers")
usage_df.write.mode("overwrite").parquet("bronze/usage")
payments_df.write.mode("overwrite").parquet("bronze/payments")
plans_df.write.mode("overwrite").parquet("bronze/plans")

Data Cleaning

In [10]:
from pyspark.sql.functions import *
customers_df.filter(col("plan_id").isNull()).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [11]:
usage_df.filter(col("data_used_gb").isNull()).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [12]:
payments_df.filter(col("amount_paid").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [13]:
payments_df.filter(col("payment_mode").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [14]:
customers_cleaned = customers_df.fillna({
    "plan_id":"UNKNOWN"
})

In [15]:
usage_cleaned = usage_df.fillna({
    "data_used_gb":0
})

payments_cleaned = payments_df.fillna({
    "amount_paid":0,
    "payment_mode":"Not Provided"
})

In [16]:
customers_cleaned.withColumn("data_quality_status",
                             when(col("plan_id")=="UNKNOWN","Missing Plan").otherwise("Valid"))

DataFrame[customer_id: int, customer_name: string, city: string, state: string, age: int, gender: string, plan_id: string, status: string, data_quality_status: string]

In [17]:
customers_cleaned = customers_cleaned.withColumn(
    "data_quality_status",
    when(col("plan_id") == "UNKNOWN","Missing Plan")
    .otherwise("Valid")
)


usage_cleaned = usage_cleaned.withColumn(
    "data_quality_status",
    when(col("data_used_gb") == 0,"Missing Usage")
    .otherwise("Valid")
)

payments_cleaned = payments_cleaned.withColumn(
    "data_quality_status",
    when(col("amount_paid") == 0,"Missing Amount")
    .otherwise("Valid")
)

In [18]:
customers_cleaned.show()
usage_cleaned.show()
payments_cleaned.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [19]:
customers_cleaned.write.mode("overwrite").parquet("silver/customers")
usage_cleaned.write.mode("overwrite").parquet("silver/usage")
payments_cleaned.write.mode("overwrite").parquet("silver/payments")

In [20]:
plans_df.show()

+-------------+--------------------+-----------+-------+------------+
|data_limit_gb|            features|monthly_fee|plan_id|   plan_name|
+-------------+--------------------+-----------+-------+------------+
|           50|{false, National,...|        499|   P101| Smart Basic|
|           75|{true, National, ...|        799|   P102|  Smart Plus|
|           25|{false, NULL, false}|        299|   P103|Budget Saver|
|          100|{true, Internatio...|       1199|   P104| Premium Max|
+-------------+--------------------+-----------+-------+------------+



In [21]:
plans_flattened = plans_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
)

In [22]:
plans_flattened.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|         NULL|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [23]:
plans_flattened = plans_flattened.fillna(
    {
        "roaming":"Not Available"
            }
)
plans_flattened.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [24]:
plans_flattened.write.mode("overwrite").parquet("silver/plans")

In [25]:
customer_plan_df = customers_cleaned.join(
    plans_flattened,"plan_id","left"
)
customer_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|              Valid| Smart Basic|        499|           50|           true|       false|     National|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|              Valid|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|              Valid|Bud

In [26]:
customer_usage_df = customers_cleaned.join(
    usage_cleaned,
    "customer_id",
    "left"
)
customer_usage_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1012|2026-02-01 00:00:00|          50|        1000|      130|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1001|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [27]:
customer_payment_df = customers_cleaned.join(
    payments_cleaned,
    customers_cleaned.customer_id == payments_cleaned.customer_id,
    "left"
)
customer_payment_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5012|        101|2026-02-01 00:00:00|        499|        Card|       Success|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|              Va

In [28]:
final_df = customers_cleaned \
    .join(plans_flattened,"plan_id","left") \
    .join(usage_cleaned,"customer_id","left") \
    .join(payments_cleaned,"customer_id","left")
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------

In [29]:
customers_cleaned.join(plans_flattened,"plan_id","left_anti").show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|              Valid|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|       Missing Plan|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+



In [30]:
usage_cleaned.join(
    customers_cleaned,
    "customer_id",
    "left_anti"
).show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|              Valid|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [31]:
payments_cleaned.join(
    customers_cleaned,
    "customer_id",
    "left_anti"
).show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



Transformations

In [32]:
final_df = final_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70,"Heavy User")
    .when(col("data_used_gb") >= 30,"Medium User")
    .otherwise("Low User")
)
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------

In [33]:
final_df = final_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000,"High Payment")
    .when(col("amount_paid") >= 500,"Medium Payment")
    .otherwise("Low Payment")
)
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+---

In [34]:
final_df = final_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15,"Medium Risk")
    .otherwise("Low Risk")
)
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+------------

In [35]:
final_df = final_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------

In [36]:
final_df = final_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0,"Yes")
    .otherwise("No")
)
final_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+----

Aggregations

In [37]:
customers_cleaned.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [38]:
final_df.groupBy("city").sum("amount_paid").show()

+---------+----------------+
|     city|sum(amount_paid)|
+---------+----------------+
|Bangalore|            3695|
|    Kochi|            1199|
|  Chennai|            1996|
|   Mumbai|             299|
|     Pune|             799|
|    Delhi|            5595|
|Hyderabad|            2295|
+---------+----------------+



In [39]:
final_df.groupBy("city") \
    .agg(
        sum("amount_paid").alias("total_revenue")
    ) \
    .show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3695|
|    Kochi|         1199|
|  Chennai|         1996|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         5595|
|Hyderabad|         2295|
+---------+-------------+



In [40]:
final_df.groupBy("plan_name") \
    .agg(
        sum("amount_paid").alias("total_revenue")
    ) \
    .show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [41]:
final_df.groupBy("plan_name").agg(sum("amount_paid").alias("revenue")).orderBy(col("revenue").desc()).show(1)

+-----------+-------+
|  plan_name|revenue|
+-----------+-------+
|Premium Max|   5995|
+-----------+-------+
only showing top 1 row


In [42]:
final_df.groupBy("usage_category").count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|   14|
|    Heavy User|    3|
|      Low User|    7|
+--------------+-----+



In [43]:
final_df.groupBy("churn_risk").count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [44]:
final_df.groupBy("plan_name").agg( sum("data_used_gb").alias("total_data_usage") ).show()

+------------+----------------+
|   plan_name|total_data_usage|
+------------+----------------+
|        NULL|            NULL|
| Smart Basic|             464|
|Budget Saver|              22|
| Premium Max|             230|
|  Smart Plus|             188|
+------------+----------------+



In [45]:
final_df.groupBy("plan_name").agg( avg("data_used_gb").alias("average_data_usage") ).show()

+------------+------------------+
|   plan_name|average_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [46]:
final_df.groupBy("city").agg( sum("call_minutes").alias("total_call_minutes") ).show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              3450|
|    Kochi|              1600|
|  Chennai|              4600|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              6600|
|Hyderabad|              4000|
+---------+------------------+



In [47]:
final_df.groupBy("state").agg( sum("sms_count").alias("total_sms") ).show()

+-----------+---------+
|      state|total_sms|
+-----------+---------+
|  Karnataka|      430|
|     Kerala|      250|
| Tamil Nadu|      620|
|      Delhi|      910|
|  Telangana|      520|
|Maharashtra|      100|
+-----------+---------+



In [48]:
final_df.filter( col("payment_status") == "Success" ).agg( sum("amount_paid").alias("total_successful_revenue") ).show()

+------------------------+
|total_successful_revenue|
+------------------------+
|                   12882|
+------------------------+



In [49]:
final_df.groupBy("city").agg( sum("amount_paid").alias("total_revenue") ).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3695|
|    Kochi|         1199|
|  Chennai|         1996|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         5595|
|Hyderabad|         2295|
+---------+-------------+



In [50]:
final_df.groupBy("plan_name").agg( sum("amount_paid").alias("total_revenue") ).show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [51]:
final_df.groupBy("payment_mode").agg( sum("amount_paid").alias("total_revenue") ).show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        NULL|         NULL|
|        Card|         6193|
|        Cash|          598|
|Not Provided|         2398|
|         UPI|         6689|
+------------+-------------+



In [52]:
final_df.groupBy("city").agg( sum("amount_paid").alias("revenue") ).orderBy(col("revenue").desc()).show(1)

+-----+-------+
| city|revenue|
+-----+-------+
|Delhi|   5595|
+-----+-------+
only showing top 1 row


Window Functions

In [53]:
from pyspark.sql.window import Window

window_spec = Window.orderBy(col("data_used_gb").desc())

rank_df = final_df.withColumn("data_usage_rank",rank().over(window_spec))

rank_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|data_usage_rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+

In [54]:
payment_window = Window.orderBy( col("amount_paid").desc() )
payment_rank_df = final_df.withColumn( "payment_rank", rank().over(payment_window) )
payment_rank_df.show()

+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|payment_rank|
+-----------+-------+-------------+---------+-----------+---+------+------+-------------------+-----------+-----------+-

In [55]:
rank_df.filter( col("data_usage_rank") <= 3 ).show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|data_usage_rank|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+----------

In [56]:
payment_rank_df.filter( col("payment_rank") <= 3 ).show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|payment_rank|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+--------------

In [57]:
city_window = Window.partitionBy( "city" ).orderBy(col("data_used_gb").desc())
top_city_df = final_df.withColumn( "rank", row_number().over(city_window) )
top_city_df.filter( col("rank") == 1 ).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+----------

In [58]:
plan_window = Window.partitionBy( "plan_name" ).orderBy(col("data_used_gb").desc())
top_plan_df = final_df.withColumn( "rank", row_number().over(plan_window) )
top_plan_df.filter( col("rank") == 1 ).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+----------

In [59]:
revenue_window = Window.orderBy("bill_month")
running_df = final_df.withColumn( "running_total_revenue", sum("amount_paid").over(revenue_window) )
running_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+---------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|running_total_revenue|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------

In [60]:
customer_window = Window.partitionBy(
    "customer_id"
).orderBy("usage_month")

lag_df = final_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb",1).over(customer_window)
)

lag_df.select("usage_month","data_used_gb","previous_month_usage").show(1000,truncate=False)

+-------------------+------------+--------------------+
|usage_month        |data_used_gb|previous_month_usage|
+-------------------+------------+--------------------+
|2026-01-01 00:00:00|45          |NULL                |
|2026-01-01 00:00:00|45          |45                  |
|2026-02-01 00:00:00|50          |45                  |
|2026-02-01 00:00:00|50          |50                  |
|2026-01-01 00:00:00|30          |NULL                |
|2026-01-01 00:00:00|30          |30                  |
|2026-02-01 00:00:00|34          |30                  |
|2026-02-01 00:00:00|34          |34                  |
|2026-01-01 00:00:00|12          |NULL                |
|2026-01-01 00:00:00|55          |NULL                |
|2026-01-01 00:00:00|55          |55                  |
|2026-02-01 00:00:00|58          |55                  |
|2026-02-01 00:00:00|58          |58                  |
|2026-01-01 00:00:00|75          |NULL                |
|2026-01-01 00:00:00|75          |75            

In [61]:
lead_df = final_df.withColumn( "next_month_usage", lead("data_used_gb",1).over(customer_window) )
lead_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|next_month_usage|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+---------

In [62]:
lag_df.filter( col("data_used_gb") > col("previous_month_usage") ).show()

+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+--------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+----------+-------------+---------------+--------------------+
|customer_id|plan_id|customer_name|     city|     state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included| roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|previous_month_usage|
+-----------+-------+-------------+---------+----------+---+------+------+-------------------+-----------+-----------+

In [63]:
customers_cleaned.createOrReplaceTempView("customers")
usage_cleaned.createOrReplaceTempView("usage")
payments_cleaned.createOrReplaceTempView("payments")
plans_flattened.createOrReplaceTempView("plans")
final_df.createOrReplaceTempView("final")

In [64]:
spark.sql(""" SELECT * FROM customers WHERE status = 'Active' """).show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|              Valid|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|              Valid|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|              Valid|
|        110|  Nisha Reddy|    Delhi|   

In [65]:
spark.sql("""SELECT city, count(*) as total_customers FROM customers GROUP BY city""").show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [66]:
spark.sql(""" SELECT plan_name, SUM(amount_paid) AS total_revenue FROM final GROUP BY plan_name """).show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [67]:
spark.sql(""" SELECT customer_name, data_used_gb FROM final WHERE data_used_gb >= 70 """).show()

+-------------+------------+
|customer_name|data_used_gb|
+-------------+------------+
|   Farhan Ali|          75|
|   Farhan Ali|          75|
|   Meera Nair|          80|
+-------------+------------+



In [68]:
spark.sql(""" SELECT customer_name, churn_risk FROM final WHERE churn_risk = 'High Risk' """).show()

+-------------+----------+
|customer_name|churn_risk|
+-------------+----------+
|   Amit Kumar| High Risk|
|   Farhan Ali| High Risk|
|   Farhan Ali| High Risk|
|  Arjun Verma| High Risk|
+-------------+----------+



In [69]:
spark.sql(""" SELECT * FROM customers WHERE plan_id = 'UNKNOWN' """).show()

+-----------+-------------+---------+---------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+---------+---+------+-------+------+-------------------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|UNKNOWN|Active|       Missing Plan|
+-----------+-------------+---------+---------+---+------+-------+------+-------------------+



In [70]:
spark.sql(""" SELECT * FROM payments WHERE payment_status IN ('Failed','Pending') """).show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|              Valid|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|              Valid|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+



In [71]:
spark.sql(""" SELECT customer_name, data_used_gb FROM final ORDER BY data_used_gb DESC LIMIT 5 """).show()

+-------------+------------+
|customer_name|data_used_gb|
+-------------+------------+
|   Meera Nair|          80|
|   Farhan Ali|          75|
|   Farhan Ali|          75|
|  Sneha Patel|          58|
|  Sneha Patel|          58|
+-------------+------------+



In [72]:
spark.sql(""" SELECT payment_mode, SUM(amount_paid) AS revenue FROM final GROUP BY payment_mode """).show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        NULL|   NULL|
|        Card|   6193|
|        Cash|    598|
|Not Provided|   2398|
|         UPI|   6689|
+------------+-------+



In [73]:
customers_cleaned = customers_cleaned.withColumnRenamed(
    "data_quality_status",
    "customer_dq_status"
)

usage_cleaned = usage_cleaned.withColumnRenamed(
    "data_quality_status",
    "usage_dq_status"
)

payments_cleaned = payments_cleaned.withColumnRenamed(
    "data_quality_status",
    "payment_dq_status"
)

final_df = customers_cleaned \
    .join(plans_flattened, "plan_id", "left") \
    .join(usage_cleaned, "customer_id", "left") \
    .join(payments_cleaned, "customer_id", "left")



In [74]:
final_df.write \
    .mode("overwrite") \
    .parquet("gold/customer_analytics")

In [75]:
final_df.write.mode("overwrite").partitionBy("usage_month").parquet("gold/customer_analytics")

In [76]:
%%writefile usage_incremental.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,62,1200,150
1017,102,2026-03,40,700,95
1018,104,2026-03,65,1400,170

Writing usage_incremental.csv


In [77]:
incremental_df = spark.read.csv( "usage_incremental.csv", header=True, inferSchema=True )

In [78]:
incremental_df.write.mode("append").parquet("silver/usage")

In [79]:
updated_usage_df = spark.read.parquet("silver/usage")
customer_usage_summary = customers_cleaned.join( updated_usage_df, "customer_id", "left" )
customer_usage_summary.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+------------------+--------+-------------------+------------+------------+---------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|customer_dq_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|
+-----------+-------------+---------+-----------+---+------+-------+--------+------------------+--------+-------------------+------------+------------+---------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|             Valid|    1016|2026-03-01 00:00:00|          62|        1200|      150|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|             Valid|    1012|2026-02-01 00:00:00|          50|        1000|      130|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|             Valid|    1001|2026-01-01 00:00:00|          45|         900|      120|
|        102|  Priya Reddy|B

In [80]:
final_df.write \
    .mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet("gold/customer_analytics")

In [81]:
before_count = usage_cleaned.count()
after_count = spark.read.parquet( "silver/usage" ).count()
print("Before:", before_count)
print("After:", after_count)

Before: 15
After: 18


In [84]:
final_df.columns

['customer_id',
 'plan_id',
 'customer_name',
 'city',
 'state',
 'age',
 'gender',
 'status',
 'customer_dq_status',
 'plan_name',
 'monthly_fee',
 'data_limit_gb',
 'unlimited_calls',
 'ott_included',
 'roaming',
 'usage_id',
 'usage_month',
 'data_used_gb',
 'call_minutes',
 'sms_count',
 'usage_dq_status',
 'payment_id',
 'bill_month',
 'amount_paid',
 'payment_mode',
 'payment_status',
 'payment_dq_status']

In [85]:
from pyspark.sql.functions import *

final_df = final_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70,"Heavy User")
    .when(col("data_used_gb") >= 30,"Medium User")
    .otherwise("Low User")
)
final_df = final_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000,"High Payment")
    .when(col("amount_paid") >= 500,"Medium Payment")
    .otherwise("Low Payment")
)
final_df = final_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15,"Medium Risk")
    .otherwise("Low Risk")
)
final_df = final_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)
final_df = final_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0,"Yes")
    .otherwise("No")
)

In [86]:
customer_usage_summary = final_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
)

customer_usage_summary.show()

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status| churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45

In [87]:

plan_report = final_df.groupBy("plan_name").agg( count_distinct("customer_id").alias("total_customers"), sum("data_used_gb").alias("total_data_usage"), avg("data_used_gb").alias("average_data_usage"), sum("amount_paid").alias("total_revenue") )
plan_report.show()

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|        NULL|              2|            NULL|              NULL|            0|
| Smart Basic|              3|             464| 51.55555555555556|         4491|
|Budget Saver|              2|              22|              11.0|          598|
| Premium Max|              2|             230|              46.0|         5995|
|  Smart Plus|              3|             188|31.333333333333332|         4794|
+------------+---------------+----------------+------------------+-------------+



In [88]:
city_report = final_df.groupBy("city").agg( count_distinct("customer_id").alias("total_customers"), sum("amount_paid").alias("total_revenue"), avg("amount_paid").alias("average_payment") )
city_report.show()

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              2|         3695|          739.0|
|    Kochi|              1|         1199|         1199.0|
|  Chennai|              1|         1996|          499.0|
|   Mumbai|              2|          299|          299.0|
|     Pune|              1|          799|          799.0|
|    Delhi|              2|         5595|         1119.0|
|Hyderabad|              3|         2295|          382.5|
+---------+---------------+-------------+---------------+



In [89]:
churn_report = final_df.select( "customer_id", "customer_name", "city", "plan_name", "payment_status", "status", "churn_risk" )
churn_report.show()

+-----------+-------------+---------+------------+--------------+--------+-----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status| churn_risk|
+-----------+-------------+---------+------------+--------------+--------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        103|   Amit Kumar|   Mumbai|Budget

In [90]:
over_usage_report = final_df.select( "customer_id", "customer_name", "plan_name", "data_used_gb", "data_limit_gb", "over_usage_gb" )
over_usage_report.show()

+-----------+-------------+------------+------------+-------------+-------------+
|customer_id|customer_name|   plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+------------+-------------+-------------+
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        102|  Priya Reddy|  Smart Plus|          34|           75|          -41|
|        102|  Priya Reddy|  Smart Plus|          34|           75|          -41|
|        102|  Priya Reddy|  Smart Plus|          30|           75|          -45|
|        102|  Priya Reddy|  Smart Plus|          30|           75|          -45|
|        103|   Amit Kumar|Budget Saver|          12|           25|          -13|
|        104|  S